# 04 - Avaliação em validação cruzada (S5, #19)

Este notebook **apenas lê** os agregados congelados em `reports/sprint-5/evaluation/` e as figuras em `reports/sprint-5/figures/`. Nada é treinado aqui; para regenerar, use `python -m src.validation_curves` e depois `python -m src.validation_plots`.

Contexto e cuidados:

- A classe positiva é 1 (`no_show=1`, o paciente **faltou**).
- As curvas vêm de folds de validação por paciente dentro do treino (`StratifiedGroupKFold`, os mesmos folds e as mesmas quatro estratégias da S4). O teste reservado não é aberto.
- Os scores de `class_weight`, `undersampling` e `smote` não são probabilidades calibradas.
- A matriz de confusão usa o limiar fixo 0,5, que não foi otimizado. Ele é diferente da regra top-20% usada no recall@20%.
- A AP (`pr_auc_ap`) é a average precision, não a área trapezoidal sob a curva precisão-recall.
- O desvio entre folds é desvio-padrão amostral, não intervalo de confiança.
- As curvas exportadas têm no máximo 501 pontos por fold e servem só para exibição; as métricas exatas foram calculadas antes da redução e não devem ser recalculadas a partir delas.
- Não há curva média nem curva agregada de todos os scores; cada linha é um fold.
- Este notebook não escolhe modelo final, não faz ajuste de hiperparâmetros nem SHAP (ver #17 e #20).

## Métricas por estratégia (média e desvio entre folds)

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
BASE = ROOT / "reports" / "sprint-5"
metadata = json.loads((BASE / "evaluation" / "metadata.json").read_text())
comparison = pd.read_csv(BASE / "evaluation" / "comparison.csv", index_col="strategy")
display(comparison)

## Figuras: ROC, precisão-recall e matrizes de confusão

In [ ]:
for name in ("roc.png", "precision_recall.png", "confusion_matrices.png"):
    display(Image(filename=str(BASE / "figures" / name)))

## Matrizes de confusão em contagens

Linhas: real (0 compareceu, 1 faltou); colunas: previsto. Soma das validações, cada agendamento de treino contado uma vez.

In [ ]:
matrices = pd.read_csv(BASE / "evaluation" / "confusion_matrices.csv")
for strategy, table in matrices.groupby("strategy", sort=False):
    print(strategy)
    display(table.pivot(index="actual", columns="predicted", values="count"))

## Leitura dos resultados

Médias dos cinco folds de validação (não são métricas do teste final):

| Estratégia | ROC-AUC | AP | Precisão (0,5) | Recall (0,5) |
|---|---:|---:|---:|---:|
| Sem balanceamento | 0,6584 | 0,3066 | 36,59% | 2,06% |
| Peso de classe | 0,6649 | 0,3073 | 31,31% | 55,73% |
| Subamostragem | 0,6644 | 0,3064 | 31,13% | 56,00% |
| SMOTENC | 0,6544 | 0,3031 | 30,57% | 55,69% |

As quatro estratégias têm AP próxima (0,3031 a 0,3073), acima da prevalência
aproximada de 20,06% no treino. Isso não demonstra superioridade estatística de
uma estratégia. No limiar fixo 0,5, o balanceamento aumenta o recall, mas também
os falsos positivos: sem balanceamento são 629 falsos positivos e 17.232 falsos
negativos; com peso de classe são 21.518 e 7.788, respectivamente, somando as
validações. Esse contraste depende do limiar e não equivale à política de contato
top-20%, nem comprova consultas recuperadas por uma intervenção.

As matrizes somam 87.720 agendamentos por estratégia, com 17.594 faltas reais;
cada linha do treino foi validada uma vez. Os 22.169 registros de teste continuam
reservados. A escolha do modelo final e o ajuste do limiar seguem fora da #19.